# Train Alapon v2

Retrains the Alapon lip-sync model on the new Redwan video, on a Colab **L4**.

**Before the first run**
1. In Google Drive, create the folder `MyDrive/alapon_v2` and upload these three files into it:
   - `alapon_redwan_v2.mp4`: the new video
   - `redwan_syncnet_best_val.pth`: the lip-sync expert trained on the *previous* Redwan video (in the repo root). It is the starting point that gets fine-tuned on the new video in step 5.
   - `alapon_code.zip`: the code (`python colab/make_code_bundle.py` builds it into `colab/`)
2. **Runtime → Change runtime type → L4 GPU** (turn on High-RAM if it is offered).
3. **Runtime → Run all.**

**If Colab disconnects:** reconnect and **Run all** again. The preprocessed data comes back from Drive and training continues from the last finished epoch. At most one epoch is lost.

Everything worth keeping (the dataset tar, the checkpoints, the logs) is written to `MyDrive/alapon_v2`, never only to the Colab machine.

## 1. Settings: the only cell you should need to edit

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/alapon_v2"   # the three uploads live here; outputs go here too
VIDEO     = "alapon_redwan_v2.mp4"
SYNCNET   = "redwan_syncnet_best_val.pth"     # previous video's expert: the fine-tuning starting point
CODE_ZIP  = "alapon_code.zip"
NAME      = "alapon_v2"                         # dataset and run name

# Lip-sync expert (SyncNet), fine-tuned on the new video before the avatar trains
SN_EPOCHS     = 30     # best validation epoch is kept, so extra epochs cannot make it worse
SN_LR         = 1e-4   # fine-tuning rate; the previous expert was trained from scratch at 1e-3
SN_BATCH      = 16     # same as the previous expert
SN_FROM_SCRATCH = False  # True: ignore the previous expert (then use SN_LR = 1e-3 and SN_EPOCHS = 100)

# Avatar (UNet)
EPOCHS        = 60     # Alapon v1 shipped its epoch-60 checkpoint (59.pth)
BATCH_SIZE    = 8      # same as v1. A bigger batch would use more GPU but trains a different model
LR            = 1e-3
VGG_WEIGHT    = 0.1    # v1 used 0.01: too weak, so the beard went smooth and a blue smear appeared under the lip
CHROMA_WEIGHT = 1.0
SYNC_WEIGHT   = 0.03
SYNC_START    = 5      # sync loss off for the first 5 epochs so the face is learned first
AMP           = False  # mixed precision is faster but changes the numerics; off for maximum quality

## 2. Connect Drive and check the machine

In [ ]:
import os, multiprocessing
from google.colab import drive
drive.mount("/content/drive")

# PyTorch >= 2.6 refuses to unpickle anything but tensors by default; these are
# our own checkpoints (landmark model, SyncNet, resume state), so allow it.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

CPUS = multiprocessing.cpu_count()
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"CPUs: {CPUS}   RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.0f} GB")

missing = [f for f in (VIDEO, SYNCNET, CODE_ZIP) if not os.path.exists(f"{DRIVE_DIR}/{f}")]
assert not missing, f"Upload {missing} to {DRIVE_DIR} first"

CKPT_DIR = f"{DRIVE_DIR}/checkpoints/{NAME}"
SN_DIR   = f"{DRIVE_DIR}/checkpoints/{NAME}_syncnet"
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(SN_DIR, exist_ok=True)
print("Checkpoints will be saved to", CKPT_DIR, "and", SN_DIR)

## 3. Unpack the code

In [ ]:
!rm -rf /content/SyncTalk_2D && unzip -q -o "{DRIVE_DIR}/{CODE_ZIP}" -d /content
%cd /content/SyncTalk_2D
!pip -q install librosa soundfile

## 4. Dataset: preprocess once, then restore from Drive

The first run takes about 10 minutes:
1. Every native frame of the 24 fps video is kept, and the audio is sped up by 25/24 to match. No frames are duplicated, and the video is never re-encoded.
2. Landmarks are detected on all CPUs.
3. Audio features are extracted.
4. The held-out test split is taken from the middle of the talking.

The result is saved to Drive as `alapon_v2_dataset.tar`, so later sessions only restore it.

In [ ]:
DS  = f"/content/SyncTalk_2D/dataset/{NAME}"
TAR = f"{DRIVE_DIR}/{NAME}_dataset.tar"

if os.path.exists(TAR):
    print("Restoring the preprocessed dataset from Drive...")
    !mkdir -p /content/SyncTalk_2D/dataset && tar -xf "{TAR}" -C /content/SyncTalk_2D/dataset
else:
    os.makedirs(DS, exist_ok=True)
    !cp "{DRIVE_DIR}/{VIDEO}" "{DS}/{NAME}.mp4"
    LM_WORKERS = max(1, min(CPUS - 1, 11))
    !python data_utils/process.py "{DS}/{NAME}.mp4" --retime --workers {LM_WORKERS}
    !python evaluation/scripts/create_holdout_manifest.py --dataset-dir "{DS}" --name {NAME} --output "{DS}/splits.json"
    # Written as .part and renamed, so a disconnect never leaves a half tar that looks complete
    !cd /content/SyncTalk_2D/dataset && tar -cf "{TAR}.part" --exclude="*.mp4" {NAME} && mv "{TAR}.part" "{TAR}"
    print("Saved", TAR)

import json
print(json.dumps(json.load(open(f"{DS}/splits.json"))["splits"], indent=1))

## 5. Lip-sync expert: fine-tune SyncNet on the new video

During avatar training, a SyncNet judges whether each generated mouth matches the audio. The one from the previous video has only seen the old recording, so it is fine-tuned here on the new video's **train split**. The test split is never shown to it, and validation uses the val split.

The first cell scores the previous expert on the new video as a baseline. In the output, **gap** = similarity of matching pairs minus that of mismatched pairs; higher means a better judge. Fine-tuning should beat the baseline gap, and `best_val.pth` keeps the best epoch.

In [ ]:
SN_WORKERS = max(2, min(8, CPUS - 2))
!python -u syncnet_328.py --dataset_dir "{DS}" --manifest "{DS}/splits.json" --asr ave \
    --save_dir "{SN_DIR}/baseline" --batch_size {SN_BATCH} --num_workers {SN_WORKERS} \
    --init "{DRIVE_DIR}/{SYNCNET}" --eval_only 2>&1 | grep -a "EVAL" | tee "{SN_DIR}/baseline_eval.txt"

In [ ]:
SN_INIT = "" if SN_FROM_SCRATCH else f'--init "{DRIVE_DIR}/{SYNCNET}"'
# --init only seeds the weights of a fresh run; once last.pth exists, --resume auto takes over.
!python -u syncnet_328.py --dataset_dir "{DS}" --manifest "{DS}/splits.json" --asr ave \
    --save_dir "{SN_DIR}" --epochs {SN_EPOCHS} --batch_size {SN_BATCH} --lr {SN_LR} \
    --num_workers {SN_WORKERS} --cache --resume auto {SN_INIT} 2>&1 | tee -a "{SN_DIR}/train_stdout.txt"

import pandas as pd, matplotlib.pyplot as plt
sl = pd.read_csv(f"{SN_DIR}/train_log.csv"); sl["gap"] = sl.val_pos_sim - sl.val_neg_sim
best = sl.loc[sl.val_loss.idxmin()]
print(open(f"{SN_DIR}/baseline_eval.txt").read().strip())
print(f"fine-tuned best: epoch {int(best.epoch)}  val {best.val_loss:.4f}  gap {best.gap:.4f}")
sl.plot(x="epoch", y=["val_loss", "gap"], subplots=True, figsize=(9, 5)); plt.show()
SYNCNET_CKPT = f"{SN_DIR}/best_val.pth"

## 6. Train the avatar

* `--cache` crops every training frame once into RAM, so the GPU is not left waiting on JPEG decoding. The samples are pixel-identical to the uncached path.
* The lip-sync expert is the fine-tuned one from step 5 (`checkpoints/alapon_v2_syncnet/best_val.pth`).
* `--resume auto` continues from `checkpoints/alapon_v2/last.pth` on Drive when it exists.
* A checkpoint is saved every 5 epochs (`4.pth`, `9.pth`, …, `59.pth`), with `last.pth` after every epoch. All saves are crash-safe.

The progress bar shows the speed. Multiply the time for one epoch by the epochs left to estimate the remaining time.

In [ ]:
WORKERS = max(2, min(8, CPUS - 2))
AMP_FLAG = "--amp" if AMP else ""
!python -u train_328.py --dataset_dir "{DS}" --manifest "{DS}/splits.json" --save_dir "{CKPT_DIR}" \
    --asr ave --use_syncnet --syncnet_checkpoint "{SN_DIR}/best_val.pth" --mask_version v2_no_jaw \
    --epochs {EPOCHS} --batchsize {BATCH_SIZE} --lr {LR} \
    --vgg_weight {VGG_WEIGHT} --chroma_weight {CHROMA_WEIGHT} \
    --sync_weight {SYNC_WEIGHT} --sync_start_epoch {SYNC_START} \
    --cache --num_workers {WORKERS} --resume auto {AMP_FLAG} 2>&1 | tee -a "{CKPT_DIR}/train_log.txt"

## 7. Check the result: loss curves and held-out frames

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f"{CKPT_DIR}/loss_log.csv")
log.plot(x="epoch", y=["l1", "vgg", "sync", "chroma"], subplots=True, figsize=(10, 8), legend=True)
plt.tight_layout(); plt.show()
print(log.tail(3).to_string(index=False))

In [ ]:
# Real vs generated on held-out test frames the model never trained on.
import sys, glob, random, torch, numpy as np
sys.path.insert(0, "/content/SyncTalk_2D")
from datasetsss_328 import MyDataset
from unet_328 import Model

ckpts = sorted(glob.glob(f"{CKPT_DIR}/[0-9]*.pth"), key=lambda p: int(os.path.basename(p)[:-4]))
assert ckpts, "no numbered checkpoint yet (the first is written after epoch 5)"
ck = ckpts[-1]
net = Model(6, mode="ave").cuda().eval()
net.load_state_dict(torch.load(ck, map_location="cuda"))

test = json.load(open(f"{DS}/splits.json"))["splits"]["test"]
ds = MyDataset(DS, "ave", start=test["start"], end=test["end"])
picks = np.linspace(0, len(ds) - 1, 4).astype(int)
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
for k, i in enumerate(picks):
    random.seed(int(i))
    x, real, a = ds[int(i)]
    with torch.no_grad():
        pred = net(x[None].cuda(), a[None].cuda())[0].float().cpu()
    for r, img in enumerate((real, pred)):
        ax[r, k].imshow(img.numpy().transpose(1, 2, 0)[100:320, 30:290, ::-1].clip(0, 1))
        ax[r, k].axis("off")
ax[0, 0].set_title("real (held-out test)", loc="left")
ax[1, 0].set_title(f"generated, {os.path.basename(ck)}", loc="left")
plt.tight_layout(); plt.show()

## 8. When training is done

Download from `MyDrive/alapon_v2/checkpoints/alapon_v2/`:
* `59.pth` (or whichever epoch looks best in step 6)
* `train_config.json`: it must sit next to the `.pth` file, because the avatar server reads the mask version from it
* `checkpoints/alapon_v2_syncnet/best_val.pth`: the new lip-sync expert, needed to evaluate or retrain later

Put both in `SyncTalk_2D/checkpoint/alapon_v2/` on your machine.